# 🧠 Model Training & Evaluation
### AI Interview Simulator — ML Pipeline

This notebook trains all 5 specialized ML models and evaluates their performance with comprehensive metrics and visualizations.

| Model | Architecture | Task |
|-------|-------------|------|
| Answer Quality | DeBERTa-v3-small | Regression (0-10) |
| Communication | DistilBERT | Multi-output (3×0-5) |
| STAR Analyzer | DeBERTa-v3-small | Multi-task (4×0-5) |
| Code Evaluator | CodeBERT | Multi-output (3×0-5) |
| Meta Scorer | XGBoost | Regression (0-10) |

In [ ]:
import sys, os, json, time
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'ml' else os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

plt.style.use('dark_background')
BRAND = '#6366f1'
COLORS = ['#6366f1', '#22c55e', '#eab308', '#ef4444', '#06b6d4', '#ec4899']

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ PyTorch {torch.__version__} | Device: {device}')
if device == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from ml.config import TrainingConfig

config = TrainingConfig()
DATA_DIR = config.data_dir
MODEL_DIR = config.model_dir

# Ensure data exists
if not os.path.exists(DATA_DIR):
    print('⚠ Training data not found. Run 01_dataset_exploration.ipynb first to generate data.')
    from ml.dataset import generate_all
    generate_all(DATA_DIR, n_per_dataset=2000)

print(f'✓ Data directory: {DATA_DIR}')
print(f'✓ Model directory: {MODEL_DIR}')
for f in os.listdir(DATA_DIR):
    size = os.path.getsize(os.path.join(DATA_DIR, f))
    print(f'  {f}: {size/1024:.1f} KB')

---
## 1. Answer Quality Model (DeBERTa-v3-small)
**Task:** Given a question-answer pair, predict a quality score (0-10).

In [ ]:
from ml.models.answer_quality import AnswerQualityTrainer

with open(os.path.join(DATA_DIR, 'answer_quality.json')) as f:
    aq_data = json.load(f)

split = int(len(aq_data) * 0.8)
aq_train, aq_val = aq_data[:split], aq_data[split:]
print(f'Train: {len(aq_train)} | Val: {len(aq_val)}')

# Reduce epochs for notebook demo (adjust as needed)
config.answer_quality.epochs = 3
config.answer_quality.batch_size = 16

print(f'\nConfig:')
print(f'  Model: {config.answer_quality.model_name}')
print(f'  Epochs: {config.answer_quality.epochs}')
print(f'  LR: {config.answer_quality.learning_rate}')
print(f'  Batch: {config.answer_quality.batch_size}')

In [ ]:
aq_trainer = AnswerQualityTrainer(config.answer_quality)

start = time.time()
aq_results = aq_trainer.train(aq_train)
elapsed = time.time() - start

print(f'\n✓ Training complete in {elapsed:.1f}s')
print(f'  Final loss: {aq_results["final_loss"]:.4f}')

aq_trainer.save(config.answer_quality.output_dir)

In [ ]:
# Evaluate on validation set
aq_trainer.model.eval()
val_preds = []
val_true = []

with torch.no_grad():
    for sample in aq_val[:100]:  # subsample for speed
        text = f"{sample['question']} [SEP] {sample['answer']}"
        tokens = aq_trainer.tokenizer(text, max_length=512, padding='max_length',
                                      truncation=True, return_tensors='pt')
        tokens = {k: v.to(aq_trainer.device) for k, v in tokens.items()}
        pred = aq_trainer.model(**tokens).item()
        val_preds.append(pred)
        val_true.append(sample['score'])

val_preds = np.array(val_preds)
val_true = np.array(val_true)

# Metrics
mse = np.mean((val_preds - val_true) ** 2)
mae = np.mean(np.abs(val_preds - val_true))
corr = np.corrcoef(val_preds, val_true)[0, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(val_true, val_preds, alpha=0.5, s=20, c=BRAND)
axes[0].plot([0, 10], [0, 10], 'r--', alpha=0.6)
axes[0].set_xlabel('True Score')
axes[0].set_ylabel('Predicted Score')
axes[0].set_title(f'Answer Quality — Pred vs True\nMSE={mse:.3f} | MAE={mae:.3f} | r={corr:.3f}')

residuals = val_preds - val_true
axes[1].hist(residuals, bins=25, color=BRAND, edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='#ef4444', linestyle='--')
axes[1].set_xlabel('Residual (Pred - True)')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

---
## 2. Communication Clarity Model (DistilBERT)
**Task:** Given answer text, predict clarity, fluency, and structure scores (0-5).

In [ ]:
from ml.models.communication import CommunicationTrainer

with open(os.path.join(DATA_DIR, 'communication.json')) as f:
    comm_data = json.load(f)

split = int(len(comm_data) * 0.8)
comm_train, comm_val = comm_data[:split], comm_data[split:]

config.communication.epochs = 3
comm_trainer = CommunicationTrainer(config.communication)

start = time.time()
comm_results = comm_trainer.train(comm_train)
elapsed = time.time() - start
print(f'\n✓ Training complete in {elapsed:.1f}s | Loss: {comm_results["final_loss"]:.4f}')
comm_trainer.save(config.communication.output_dir)

In [ ]:
# Evaluate Communication model
comm_trainer.model.eval()
comm_preds = {'clarity': [], 'fluency': [], 'structure': []}
comm_true = {'clarity': [], 'fluency': [], 'structure': []}

with torch.no_grad():
    for sample in comm_val[:100]:
        tokens = comm_trainer.tokenizer(sample['text'], max_length=384, padding='max_length',
                                        truncation=True, return_tensors='pt')
        tokens = {k: v.to(comm_trainer.device) for k, v in tokens.items()}
        preds = comm_trainer.model(**tokens).squeeze(0).cpu().numpy()
        for idx, dim in enumerate(['clarity', 'fluency', 'structure']):
            comm_preds[dim].append(preds[idx])
            comm_true[dim].append(sample[dim])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for idx, (dim, color) in enumerate(zip(['clarity', 'fluency', 'structure'], COLORS[:3])):
    t, p = np.array(comm_true[dim]), np.array(comm_preds[dim])
    mse = np.mean((p - t) ** 2)
    axes[idx].scatter(t, p, alpha=0.5, s=20, c=color)
    axes[idx].plot([0, 5], [0, 5], 'r--', alpha=0.6)
    axes[idx].set_xlabel('True')
    axes[idx].set_ylabel('Predicted')
    axes[idx].set_title(f'{dim.title()} — MSE={mse:.3f}')

plt.suptitle('Communication Model — Predictions vs True', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. STAR Behavioral Analyzer (DeBERTa-v3-small)
**Task:** Given a behavioral answer, predict S/T/A/R component scores (0-5 each).

In [ ]:
from ml.models.star_analyzer import StarAnalyzerTrainer

with open(os.path.join(DATA_DIR, 'star_analyzer.json')) as f:
    star_data = json.load(f)

split = int(len(star_data) * 0.8)
star_train, star_val = star_data[:split], star_data[split:]

config.star_analyzer.epochs = 3
star_trainer = StarAnalyzerTrainer(config.star_analyzer)

start = time.time()
star_results = star_trainer.train(star_train)
elapsed = time.time() - start
print(f'\n✓ Training complete in {elapsed:.1f}s | Loss: {star_results["final_loss"]:.4f}')
star_trainer.save(config.star_analyzer.output_dir)

In [ ]:
# Evaluate STAR model
star_trainer.model.eval()
star_dims = ['situation_score', 'task_score', 'action_score', 'result_score']
star_labels = ['Situation', 'Task', 'Action', 'Result']
star_preds = {d: [] for d in star_dims}
star_true_vals = {d: [] for d in star_dims}

with torch.no_grad():
    for sample in star_val[:100]:
        tokens = star_trainer.tokenizer(sample['text'], max_length=512, padding='max_length',
                                        truncation=True, return_tensors='pt')
        tokens = {k: v.to(star_trainer.device) for k, v in tokens.items()}
        preds = star_trainer.model(**tokens).squeeze(0).cpu().numpy()
        for idx, dim in enumerate(star_dims):
            star_preds[dim].append(preds[idx])
            star_true_vals[dim].append(sample[dim])

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
star_colors = ['#3b82f6', '#8b5cf6', '#22c55e', '#f59e0b']
for idx, (dim, label, color) in enumerate(zip(star_dims, star_labels, star_colors)):
    ax = axes[idx // 2][idx % 2]
    t, p = np.array(star_true_vals[dim]), np.array(star_preds[dim])
    mse = np.mean((p - t) ** 2)
    ax.scatter(t, p, alpha=0.5, s=20, c=color)
    ax.plot([0, 5], [0, 5], 'r--', alpha=0.6)
    ax.set_xlabel('True')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{label} — MSE={mse:.3f}')

plt.suptitle('STAR Analyzer — Predictions vs True', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Code Quality Evaluator (CodeBERT)
**Task:** Given source code, predict quality, efficiency, and style scores (0-5 each).

In [ ]:
from ml.models.code_evaluator import CodeEvaluatorTrainer

with open(os.path.join(DATA_DIR, 'code_evaluator.json')) as f:
    code_data = json.load(f)

split = int(len(code_data) * 0.8)
code_train, code_val = code_data[:split], code_data[split:]

config.code_evaluator.epochs = 3
config.code_evaluator.batch_size = 8
code_trainer = CodeEvaluatorTrainer(config.code_evaluator)

start = time.time()
code_results = code_trainer.train(code_train)
elapsed = time.time() - start
print(f'\n✓ Training complete in {elapsed:.1f}s | Loss: {code_results["final_loss"]:.4f}')
code_trainer.save(config.code_evaluator.output_dir)

In [ ]:
# Evaluate Code model
code_trainer.model.eval()
code_dims = ['quality', 'efficiency', 'style']
code_preds = {d: [] for d in code_dims}
code_true_vals = {d: [] for d in code_dims}

with torch.no_grad():
    for sample in code_val[:100]:
        tokens = code_trainer.tokenizer(sample['code'], max_length=512, padding='max_length',
                                        truncation=True, return_tensors='pt')
        tokens = {k: v.to(code_trainer.device) for k, v in tokens.items()}
        preds = code_trainer.model(**tokens).squeeze(0).cpu().numpy()
        for idx, dim in enumerate(code_dims):
            code_preds[dim].append(preds[idx])
            code_true_vals[dim].append(sample[dim])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
code_colors = ['#06b6d4', '#22c55e', '#ec4899']
for idx, (dim, color) in enumerate(zip(code_dims, code_colors)):
    t, p = np.array(code_true_vals[dim]), np.array(code_preds[dim])
    mse = np.mean((p - t) ** 2)
    axes[idx].scatter(t, p, alpha=0.5, s=20, c=color)
    axes[idx].plot([0, 5], [0, 5], 'r--', alpha=0.6)
    axes[idx].set_xlabel('True')
    axes[idx].set_ylabel('Predicted')
    axes[idx].set_title(f'{dim.title()} — MSE={mse:.3f}')

plt.suptitle('Code Evaluator — Predictions vs True', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Meta Scorer (XGBoost)
**Task:** Aggregate all model outputs into a final interview score (0-10).

In [ ]:
from ml.models.meta_scorer import MetaScorer

with open(os.path.join(DATA_DIR, 'meta_scorer.json')) as f:
    meta_data = json.load(f)

split = int(len(meta_data) * 0.8)
meta_train, meta_val = meta_data[:split], meta_data[split:]
print(f'Train: {len(meta_train)} | Val: {len(meta_val)}')

meta_scorer = MetaScorer(config.meta_scorer)

start = time.time()
meta_results = meta_scorer.train(meta_train, meta_val)
elapsed = time.time() - start
print(f'\n✓ Training complete in {elapsed:.1f}s')
meta_scorer.save(config.meta_scorer.output_dir)

In [ ]:
# Feature importance visualization
importance = meta_scorer.feature_importance()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Feature importance bar chart
feat_names = list(importance.keys())
feat_vals = list(importance.values())
colors_fi = [BRAND if i < 5 else '#4b5563' for i in range(len(feat_names))]
axes[0].barh(feat_names[::-1], feat_vals[::-1], color=colors_fi[::-1], edgecolor='white')
axes[0].set_xlabel('Importance')
axes[0].set_title('Feature Importance (XGBoost)', fontweight='bold')

# Predictions vs true on validation
meta_true = np.array([d['final_score'] for d in meta_val])
meta_pred = np.array([meta_scorer.predict(d) for d in meta_val])
mse = np.mean((meta_pred - meta_true) ** 2)
mae = np.mean(np.abs(meta_pred - meta_true))
r = np.corrcoef(meta_pred, meta_true)[0, 1]

axes[1].scatter(meta_true, meta_pred, alpha=0.3, s=10, c=BRAND)
axes[1].plot([0, 10], [0, 10], 'r--', alpha=0.6)
axes[1].set_xlabel('True Score')
axes[1].set_ylabel('Predicted Score')
axes[1].set_title(f'Meta Scorer — Pred vs True\nMSE={mse:.3f} | MAE={mae:.3f} | r={r:.3f}', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 6. Training Summary & Model Comparison

In [ ]:
# Aggregate all results
summary = pd.DataFrame([
    {'Model': 'Answer Quality', 'Architecture': 'DeBERTa-v3-small', 'Task': 'Regression',
     'Final Loss': aq_results.get('final_loss', None), 'Output Dims': 1},
    {'Model': 'Communication', 'Architecture': 'DistilBERT', 'Task': 'Multi-output',
     'Final Loss': comm_results.get('final_loss', None), 'Output Dims': 3},
    {'Model': 'STAR Analyzer', 'Architecture': 'DeBERTa-v3-small', 'Task': 'Multi-task',
     'Final Loss': star_results.get('final_loss', None), 'Output Dims': 4},
    {'Model': 'Code Evaluator', 'Architecture': 'CodeBERT', 'Task': 'Multi-output',
     'Final Loss': code_results.get('final_loss', None), 'Output Dims': 3},
    {'Model': 'Meta Scorer', 'Architecture': 'XGBoost', 'Task': 'Regression',
     'Final Loss': meta_results.get('train_mse', None), 'Output Dims': 1},
])

print('\n' + '═' * 70)
print('  📊 TRAINING SUMMARY')
print('═' * 70)
print(summary.to_string(index=False))
print('═' * 70)

# Visualize final losses
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(summary['Model'], summary['Final Loss'], color=COLORS[:5], edgecolor='white', width=0.6)
ax.set_ylabel('Final Loss (MSE)')
ax.set_title('Model Training — Final Loss Comparison', fontweight='bold', fontsize=13)
for i, (_, row) in enumerate(summary.iterrows()):
    if row['Final Loss'] is not None:
        ax.text(i, row['Final Loss'] + 0.01, f'{row["Final Loss"]:.4f}', ha='center', fontsize=9, color='white')
plt.tight_layout()
plt.show()

In [ ]:
# Check model sizes on disk
print('\n💾 Saved Model Sizes:')
for model_name in ['answer_quality', 'communication', 'star_analyzer', 'code_evaluator', 'meta_scorer']:
    model_path = os.path.join(MODEL_DIR, model_name)
    if os.path.exists(model_path):
        total = sum(os.path.getsize(os.path.join(model_path, f)) for f in os.listdir(model_path))
        print(f'  {model_name:20s} → {total / 1024 / 1024:.1f} MB')
    else:
        print(f'  {model_name:20s} → not saved')

---
## 7. Inference Demo
Test the trained models with sample inputs.

In [ ]:
from ml.inference import get_ml_service

service = get_ml_service()

print('Available Models:')
for name, loaded in service.get_available_models().items():
    status = '✓ Loaded' if loaded else '✗ Not available'
    print(f'  {name:20s} {status}')

In [ ]:
# Test predictions
print('\n🔮 Sample Predictions:')
print('─' * 50)

# Answer quality
q = 'What is a hash table?'
good_a = 'A hash table is a data structure that maps keys to values using a hash function. It provides O(1) average lookup.'
bad_a = 'I think its some kind of table thing.'

score_good = service.predict_answer_quality(q, good_a)
score_bad = service.predict_answer_quality(q, bad_a)
print(f'Answer Quality (good): {score_good}/10')
print(f'Answer Quality (bad):  {score_bad}/10')

# Communication
comm = service.predict_communication(good_a)
print(f'\nCommunication: {comm}')

# STAR
star_answer = 'At my previous company, we had a critical outage. My task was to coordinate the incident response. I set up a war room and assigned roles. As a result, we resolved it in 2 hours.'
star_scores = service.predict_star(star_answer)
print(f'\nSTAR Scores: {star_scores}')

# Code quality
code = 'def fib(n):\n    if n <= 1: return n\n    return fib(n-1) + fib(n-2)'
code_scores = service.predict_code_quality(code)
print(f'\nCode Quality: {code_scores}')